<div style="display: flex; justify-content: flex-start; align-items: center;">
    <a href="https://colab.research.google.com/github/msfasha/307307-BI-Methods-Generative-AI/blob/main/20261/Module%206%20-%20Introduction%20to%20CrewAI/CrewAI_Basics_lab.ipynb" target="_blank">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 25px;">
    </a>
</div>

<div style="background-color:#F5EDF4; border-top:3px solid #A83296; padding:16px; border-radius:6px">
<h1 style="color:#493347; margin:0">Lab 1: CrewAI Basics</h1>
<p style="color:#56616D; margin:6px 0 0 0">307307 BI Methods for Generative AI - University of Petra</p>
</div>

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Objectives</h2>
</div>

Build a small team of two AI agents and learn the six CrewAI building blocks:

| Building block | Meaning | In this lab |
|---|---|---|
| **LLM** | The language model behind the agents | Gemini |
| **Agent** | An AI worker with a role, goal and backstory | Market researcher, strategist |
| **Task** | One unit of work, given to one agent | Market overview, recommendation |
| **Crew** | The agents and tasks, run together | `student_app_crew` |
| **Process** | The order in which tasks run | `Process.sequential` |
| **Tool** | A Python function an agent can call | Product brief lookup |

**The example.** We ask a two-agent team whether to build a study-planner app for university students. One agent writes a market overview; the other turns it into a recommendation.

> The agents do not look anything up online. Their answers come from what the model already knows, so treat them as illustrative, not as verified research.

Next, `CrewAI_Intro_lab.ipynb` builds a larger team with real data.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Setup</h2>
</div>

Install CrewAI with Gemini support, then give it your Gemini API key. In Google Colab, store the key as `GOOGLE_API_KEY` in the Secrets panel (the key icon on the left). Anywhere else, the cell asks you to paste it. Get a key from [Google AI Studio](https://aistudio.google.com/app/apikey), and never put a key inside a notebook you share.

In [ ]:
# Install CrewAI with Gemini support.
%pip install -q "crewai[google-genai]"

In [ ]:
import os, getpass

# Read the API key from Colab Secrets if available, otherwise ask for it.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

# Turn off anonymous usage reporting.
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

print("API key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Building block 1: LLM</h2>
</div>

CrewAI talks to many providers. The prefix `gemini/` selects Google Gemini. If you get a "model not found" error, change only this line to a model your key can use.

In [ ]:
from crewai import LLM

llm = LLM(model="gemini/gemini-3.8-flash")   # change this line if your key cannot see this model

# Quick test: one direct question to the model, no agents yet.
print(llm.call("Say hello in one short sentence."))

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Building block 2: Agent</h2>
</div>

An **Agent** is an AI worker. Three sentences describe it, and CrewAI turns them into the instructions the LLM receives:

- `role`: who the agent is.
- `goal`: what it should achieve.
- `backstory`: background that guides its behaviour.

Write them like a job description. `verbose=True` prints the agent's thinking so you can follow it.

In [ ]:
from crewai import Agent

market_researcher = Agent(
    role="Market Research Analyst",
    goal="Prepare a clear, balanced overview of a product opportunity.",
    backstory=(
        "You are a careful market research analyst. "
        "You separate assumptions from facts and write for a business audience."
    ),
    llm=llm,
    verbose=True,
)

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Building block 3: Task</h2>
</div>

A **Task** describes the work: a `description` (what to do), an `expected_output` (what a good answer looks like) and the `agent` who does it.

In [ ]:
from crewai import Task

research_task = Task(
    description=(
        "Analyse the opportunity for a mobile app that helps university students "
        "organise their study schedules. Cover target users, needs, alternatives, "
        "opportunities and risks. Label assumptions clearly. "
        "Do not invent statistics or claim to have done live research."
    ),
    expected_output=(
        "A concise market overview with sections: target users, needs, "
        "alternatives, opportunity, risks, assumptions."
    ),
    agent=market_researcher,
)

### A second agent and task

The strategist receives the researcher's result through `context=[research_task]`. This is how one task passes its work to the next.

In [ ]:
business_strategist = Agent(
    role="Business Strategy Consultant",
    goal="Turn the market overview into a practical business recommendation.",
    backstory=(
        "You are a pragmatic consultant who looks at customer value, "
        "differentiation, feasibility and business risk."
    ),
    llm=llm,
    verbose=True,
)

strategy_task = Task(
    description=(
        "Using the market overview, recommend whether to build the student "
        "study-planning app. Propose an MVP with three core features, one "
        "differentiator, three next steps and the key risks."
    ),
    expected_output=(
        "A short recommendation: go/no-go view, three MVP features, "
        "differentiator, risks and next steps."
    ),
    agent=business_strategist,
    context=[research_task],     # receives the result of the first task
)

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Building blocks 4 and 5: Crew and Process</h2>
</div>

A **Crew** puts agents and tasks together. Its **Process** says how the tasks run:

- `Process.sequential`: one task after another, in the order given (an assembly line).
- `Process.hierarchical`: a manager agent hands out the work (used in the next lab).

We use sequential: research first, then strategy.

In [ ]:
from crewai import Crew, Process

student_app_crew = Crew(
    agents=[market_researcher, business_strategist],
    tasks=[research_task, strategy_task],
    process=Process.sequential,
    verbose=True,
)

### Run the crew

`kickoff` starts the work. It sends requests to Gemini, so it uses some of your API quota and takes a minute. (`await ...kickoff_async()` is the version that works directly in a notebook cell.)

In [ ]:
result = await student_app_crew.kickoff_async()

print("\n===== FINAL OUTPUT =====\n")
print(result)

### What happened

1. The Market Research Analyst completed `research_task`.
2. The Business Strategy Consultant received that result through `context` and completed `strategy_task`.
3. The crew returned the last task's result.

```text
Crew = Agents + Tasks + Process

Researcher --(Task 1: market overview)--> Strategist --(Task 2: recommendation)--> Final result
```

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Building block 6: Tool</h2>
</div>

A language model only produces text. A **Tool** is a Python function the agent can call to do more: calculate, read a file, query a database or search the web. The agent reads the tool's name and the description in triple quotes, decides when to use it, and gets the result back.

This teaching tool returns a fixed sample brief. It is not a live data source.

In [ ]:
from crewai.tools import tool

@tool("Product brief lookup")
def get_product_brief(product_name: str) -> str:
    """Return a sample product brief for a student study-planning app."""
    if "study" in product_name.lower() or "student" in product_name.lower():
        return (
            "Sample brief: university students may need deadline tracking, "
            "weekly planning and reminders. This is illustrative input, "
            "not verified market research."
        )
    return "No sample brief is available for this product."

# A tool is an ordinary function: test it directly, without any AI.
print(get_product_brief.run("student study planner"))

Give the tool to an agent with `tools=[...]`. The agent decides when to use it, and the log shows each call.

In [ ]:
tool_agent = Agent(
    role="Product Analyst",
    goal="Use the product brief to summarise likely user needs.",
    backstory="You are a concise analyst who uses the tools you are given.",
    llm=llm,
    tools=[get_product_brief],     # the tool, given to the agent
    verbose=True,
)

tool_task = Task(
    description=(
        "Use the Product brief lookup tool for 'student study planner'. "
        "Summarise what it returns and state its limitations."
    ),
    expected_output="A short summary based on the tool result, including limitations.",
    agent=tool_agent,
)

tool_crew = Crew(agents=[tool_agent], tasks=[tool_task], process=Process.sequential, verbose=True)

tool_result = await tool_crew.kickoff_async()
print(tool_result)

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Knowledge check</h2>
</div>

1. What is the difference between an Agent and a Task?
2. What do `role`, `goal` and `backstory` describe?
3. What does a Crew contain?
4. What is the difference between a sequential and a hierarchical process?
5. Why might an agent need a Tool?
6. Why do we call the market overview illustrative rather than verified research?

<details>
<summary>Suggested answers</summary>

1. An Agent does work; a Task defines the work to be done.
2. The agent's job, its objective, and background that guides its behaviour.
3. Agents and tasks, run under a process.
4. Sequential runs tasks in a fixed order; hierarchical uses a manager agent to hand out the work.
5. To act or get information beyond generating text (calculate, read data, search).
6. Because the agents have no live data source; their answers come from the model's general knowledge.
</details>

**Try it yourself.** Change the product idea to something from your own life or studies, rerun the crew, and compare the results.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Summary</h2>
</div>

| Building block | Meaning | In this lab |
|---|---|---|
| LLM | Language model behind the agents | Gemini |
| Agent | AI worker with role, goal, backstory | Researcher, strategist |
| Task | One unit of work | Market overview, recommendation |
| Crew | Agents and tasks run together | `student_app_crew` |
| Process | Order of the work | `Process.sequential` |
| Tool | Function an agent can call | Product brief lookup |

**Next:** `CrewAI_Intro_lab.ipynb` builds a bigger team for a coffee shop, adding real data tools, a knowledge base, memory, quality checks and a fixed workflow.